In [3]:
!pip install pyecharts
import pandas as pd
from pyecharts.charts import Parallel
from pyecharts.options import ParallelAxisOpts, TitleOpts

# 读取数据
df = pd.read_excel("某次考试成绩.xlsx", sheet_name="Sheet2 (2)")

# 选取部分维度用于展示
selected_columns = ["新闻报道", "日常对话", "日常短文", "阅读1", "阅读2", "语法知识", "语言应用", "结构", "语言", "交互"]
df_selected = df[selected_columns].dropna().reset_index(drop=True)

# 构建平行坐标维度配置
parallel_axis = [
    ParallelAxisOpts(dim=i, name=col) for i, col in enumerate(selected_columns)
]

# 数据转为 list 格式
data = df_selected.values.tolist()

# 构建平行坐标图
parallel = (
    Parallel()
    .add_schema(schema=parallel_axis)
    .add("学生成绩", data)
    .set_global_opts(title_opts=TitleOpts(title="学生多维成绩平行坐标图"))
)

# 渲染为 HTML 文件
parallel.render("学生成绩_parallel.html")
print("图表已保存为：学生成绩_parallel.html")


图表已保存为：学生成绩_parallel.html


In [2]:
import pandas as pd
from pyecharts.charts import Bar, Radar, Tab
from pyecharts.options import TitleOpts
from pyecharts.globals import ThemeType

# 读取数据
df = pd.read_excel("某次考试成绩.xlsx", sheet_name="Sheet2 (2)")

# ---------------------- 图1：Bar - 前10位学生阅读模块得分 ----------------------
bar = Bar(init_opts={"theme": ThemeType.LIGHT})
read_cols = ["阅读1", "阅读2", "阅读3", "阅读4"]

# 只取前10个学生
df_bar = df[read_cols].head(10)
bar.add_xaxis([f"学生{i+1}" for i in range(len(df_bar))])
for col in read_cols:
    bar.add_yaxis(col, df_bar[col].fillna(0).tolist())

bar.set_global_opts(title_opts=TitleOpts(title="前10位学生的阅读模块得分（柱状图）"))

# ---------------------- 图2：Radar - 前5个学院语法与语言应用平均得分 ----------------------
df_radar = df[["学院", "语法知识", "语言应用"]].dropna()
grouped = df_radar.groupby("学院")[["语法知识", "语言应用"]].mean()
top5 = grouped.head(5)

schema = [
    {"name": "语法知识", "max": 10},
    {"name": "语言应用", "max": 10},
]

radar = Radar()
radar.add_schema(schema=schema)
for idx, row in top5.iterrows():
    radar.add(idx, [row.tolist()])

radar.set_global_opts(title_opts=TitleOpts(title="前5个学院语法与语言应用雷达图"))

# ---------------------- Tab 合并 ----------------------
tab = Tab()
tab.add(bar, "阅读模块前10学生")
tab.add(radar, "前5学院雷达图")

# 渲染
tab.render("考试成绩_Tab简化版.html")
print("✅ 简化版 Tab 图表已生成：考试成绩_Tab简化版.html")


✅ 简化版 Tab 图表已生成：考试成绩_Tab简化版.html


In [6]:
import pandas as pd
from pyecharts.charts import Bar, Timeline
from pyecharts.options import TitleOpts
from pyecharts.globals import ThemeType

# 读取数据
df = pd.read_excel("某次考试成绩.xlsx", sheet_name="Sheet2 (2)")

# 选择分析模块
score_cols = ["新闻报道", "日常对话", "阅读1", "阅读2", "阅读3", "阅读4", "语法知识"]
df_filtered = df[["学院"] + score_cols].dropna()

# 构造 Timeline
tl = Timeline(init_opts={"theme": ThemeType.LIGHT})
academies = df_filtered["学院"].unique()

for academy in academies:
    sub_df = df_filtered[df_filtered["学院"] == academy]
    means = sub_df[score_cols].mean().round(2).tolist()

    bar = (
        Bar()
        .add_xaxis(score_cols)
        .add_yaxis("平均得分", means)
        .set_global_opts(title_opts=TitleOpts(title=f"{academy} 平均成绩"))
    )
    tl.add(bar, time_point=academy)

# 渲染
tl.render("考试成绩_Timeline按学院展示.html")
print("✅ Timeline 图表已生成：考试成绩_Timeline按学院展示.html")


✅ Timeline 图表已生成：考试成绩_Timeline按学院展示.html


In [8]:
import pandas as pd
from pyecharts.charts import Bar, Line, Grid
from pyecharts.options import TitleOpts, LegendOpts
from pyecharts.globals import ThemeType

from pyecharts.options import InitOpts

# 读取数据
df = pd.read_excel("某次考试成绩.xlsx", sheet_name="Sheet2 (2)")

# 选取维度
modules = ["新闻报道", "日常对话", "日常短文", "阅读1", "阅读2", "语法知识", "语言应用"]

# ----------------- 左图：各学院语法知识平均分（Bar） -----------------
df_bar = df[["学院", "语法知识"]].dropna()
df_bar_grouped = df_bar.groupby("学院").mean().reset_index()

bar = Bar(init_opts={"theme": ThemeType.LIGHT, "width": "800px", "height": "400px"})
bar.add_xaxis(df_bar_grouped["学院"].tolist())
bar.add_yaxis("语法知识平均分", df_bar_grouped["语法知识"].round(2).tolist())
bar.set_global_opts(
    title_opts=TitleOpts(title="各学院语法知识平均得分"),
    legend_opts=LegendOpts(pos_top="10%")
)

# ----------------- 右图：第1位学生的各模块得分（Line） -----------------
student = df.iloc[0]  # 第1位学生
line = Line()
line.add_xaxis(modules)
line.add_yaxis("学生1得分", [student[col] for col in modules])
line.set_global_opts(title_opts=TitleOpts(title="第1位学生的模块得分趋势"))

# ----------------- 使用 Grid 组合图表 -----------------
grid = Grid(init_opts=InitOpts(width="1000px", height="500px", theme=ThemeType.LIGHT))
grid.add(bar, grid_opts={"pos_left": "5%", "pos_right": "55%"})
grid.add(line, grid_opts={"pos_left": "55%", "pos_right": "5%"})

# 渲染输出
grid.render("考试成绩_Grid图表展示.html")
print("✅ Grid 图表已生成：考试成绩_Grid图表展示.html")


✅ Grid 图表已生成：考试成绩_Grid图表展示.html


In [12]:
import pandas as pd
from pyecharts.charts import Bar, Line, Radar, Parallel, Page
from pyecharts.options import TitleOpts, PageLayoutOpts, InitOpts
from pyecharts.globals import ThemeType

# 读取数据
df = pd.read_excel("某次考试成绩.xlsx", sheet_name="Sheet2 (2)")
modules = ["新闻报道", "日常对话", "日常短文", "阅读1", "阅读2", "语法知识", "语言应用"]

# ----------------- 图表1：Bar - 各学院语法知识平均分 -----------------
df_bar = df[["学院", "语法知识"]].dropna()
df_bar_grouped = df_bar.groupby("学院").mean().reset_index()

bar = (
    Bar()
    .add_xaxis(df_bar_grouped["学院"].tolist())
    .add_yaxis("语法知识平均分", df_bar_grouped["语法知识"].round(2).tolist())
    .set_global_opts(title_opts=TitleOpts(title="各学院语法知识平均得分"))
)

# ----------------- 图表2：Line - 第1位学生模块得分趋势 -----------------
student = df.iloc[0]
line = (
    Line()
    .add_xaxis(modules)
    .add_yaxis("学生1得分", [student[col] for col in modules])
    .set_global_opts(title_opts=TitleOpts(title="第1位学生的模块得分趋势"))
)

# ----------------- 图表3：Radar - 各学院语法与语言应用能力 -----------------
grouped = df.groupby("学院")[["语法知识", "语言应用"]].mean().dropna()
schema = [
    {"name": "语法知识", "max": 10},
    {"name": "语言应用", "max": 10},
]
radar = Radar()
radar.add_schema(schema=schema)
for idx, row in grouped.iterrows():
    radar.add(idx, [row.tolist()])
radar.set_global_opts(title_opts=TitleOpts(title="各学院语法与语言应用雷达图"))

# ----------------- 图表4：Parallel - 学生成绩多维分布 -----------------
selected_cols = ["新闻报道", "日常对话", "日常短文", "阅读1", "阅读2", "语法知识", "语言应用", "结构", "语言", "交互"]
df_parallel = df[selected_cols].dropna().reset_index(drop=True)
parallel = (
    Parallel()
    .add_schema([{"dim": i, "name": col} for i, col in enumerate(selected_cols)])
    .add("学生成绩", df_parallel.values.tolist())
    .set_global_opts(title_opts=TitleOpts(title="学生成绩平行坐标图"))
)

# ----------------- 页面组合 Page -----------------
page = Page(page_title="考试成绩可视化报告")
page.add(bar, line, radar, parallel)
page.render("考试成绩_Page页面展示.html")

# ----------------- 输出 HTML -----------------
page.render("考试成绩_Page页面展示.html")
print("✅ 图表已生成：考试成绩_Page页面展示.html")



✅ 图表已生成：考试成绩_Page页面展示.html
